# Convolutional Neural Network (CNN) — MNIST

A CNN learns to detect **spatial patterns** in images (edges, shapes, textures) using filters that slide across the image — much more powerful than a plain neural net for vision tasks.

**Architecture:**
```
Input (1×28×28)
  → Conv2d(32 filters, 3×3) → ReLU → MaxPool(2×2)   # detects low-level features
  → Conv2d(64 filters, 3×3) → ReLU → MaxPool(2×2)   # detects high-level features
  → Flatten
  → Linear(128) → ReLU → Dropout(0.2)
  → Linear(10)                                        # class scores
```
**Key concepts:**
- **Conv2d** — applies learnable filters across the image to detect features
- **MaxPool** — downsamples by taking the max in each 2×2 region, reducing size
- **Flatten** — converts 2D feature maps into a 1D vector for the classifier

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Load Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f'Train: {len(train_data)} | Test: {len(test_data)}')

## 2. Define the CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1×28×28 → 32×13×13
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1 input channel (grayscale), 32 filters
            nn.ReLU(),
            nn.MaxPool2d(2),                              # halves spatial size: 28→14

            # Block 2: 32×14×14 → 64×7×7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),           # 64×7×7 = 3136
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 3. Visualize What the Filters Learn

Before training, filters are random. After training, they'll detect meaningful features like edges and curves.

In [ ]:
def show_filters(model, title):
    filters = model.features[0].weight.data.cpu()  # first Conv2d layer
    fig, axes = plt.subplots(4, 8, figsize=(10, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(filters[i, 0], cmap='gray')
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_filters(model, 'Filters BEFORE training (random)')

## 4. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5
losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

print('\nTraining complete!')

## 5. Filters After Training

In [ ]:
show_filters(model, 'Filters AFTER training (learned features)')

## 6. Plot Training Loss

In [ ]:
plt.plot(range(1, EPOCHS+1), losses, marker='o', color='steelblue')
plt.title('CNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## 7. Evaluate

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(model(images), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'CNN Test Accuracy: {accuracy:.2f}%')
print(f'(Plain neural net was ~97% — CNN should be ~99%)')

## 8. Visualize Feature Maps

See what the CNN "sees" at each layer for a single image.

In [ ]:
# Pick one test image
images, labels = next(iter(test_loader))
sample = images[0:1].to(device)  # shape: 1×1×28×28

# Show original image
plt.figure(figsize=(2, 2))
plt.imshow(images[0].squeeze(), cmap='gray')
plt.title(f'Input: digit {labels[0].item()}')
plt.axis('off')
plt.show()

# Show feature maps after Conv layer 1
model.eval()
with torch.no_grad():
    # Pass through just the first two layers (Conv + ReLU)
    out = model.features[:2](sample)  # shape: 1×32×28×28

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(out[0, i].cpu(), cmap='viridis')
    ax.axis('off')
plt.suptitle('Feature Maps after Conv Layer 1 (32 filters)')
plt.tight_layout()
plt.show()

## Next Steps

- Try **FashionMNIST** — same format, much harder (clothing categories)
- Try **CIFAR-10** — color images (RGB), 10 object categories
- Add **Batch Normalization** (`nn.BatchNorm2d`) for faster, more stable training
- Use a pretrained model with **Transfer Learning** (ResNet, VGG)